<a href="https://colab.research.google.com/github/sainudheenp/pyLab/blob/main/Qwen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Thu May 14 14:05:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.8 MB/s eta 0:00:00


In [1]:
!pip install -U transformers accelerate bitsandbytes

In [3]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

import torch

model_name = "Qwen/Qwen2.5-Coder-7B-Instruct"

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.float16
)

print("Qwen loaded successfully!")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen loaded successfully!


In [4]:
prompt = "Write a Python Flask API for todo app"

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=300
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write a Python Flask API for todo app
assistant
Sure! Below is a simple example of a Flask API for a to-do app. This API allows you to create, read, update, and delete tasks.

First, make sure you have Flask installed. If not, you can install it using pip:

```sh
pip install Flask
```

Now, let's create the Flask application:

```python
from flask import Flask, request, jsonify

app = Flask(__name__)

# In-memory storage for simplicity
tasks = [
    {'id': 1, 'title': 'Buy groceries', 'completed': False},
    {'id': 2, 'title': 'Read a book', 'completed': True}
]

@app.route('/tasks', methods=['GET'])
def get_tasks():
    return jsonify({'tasks': tasks})

@app.route('/tasks/<int:task_id>', methods=['GET'])
def get_task(task_id):
    task = next((task for task in tasks if task['id'] == task_id), None)
    if task:
        return jsonify(task)
    else:
        return jsonify({'error': 'Task not found'}), 40

In [5]:
!pip install -q fastapi uvicorn pyngrok nest_asyncio

In [6]:
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
from threading import Thread
import nest_asyncio

nest_asyncio.apply()

app = FastAPI()

class RequestData(BaseModel):
    prompt: str

@app.post("/generate")
def generate(data: RequestData):

    messages = [
        {"role": "user", "content": data.prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        top_p=0.9
    )

    response = outputs[0][inputs.input_ids.shape[1]:]

    reply = tokenizer.decode(
        response,
        skip_special_tokens=True
    )

    return {"response": reply}

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = Thread(target=run)
thread.start()

In [8]:
from pyngrok import ngrok

ngrok.set_auth_token("3DidCcRuIgtLzGGes8OMC71MikT_4RJJUSeJN9h4Bf96qLud2")
public_url = ngrok.connect(8000)
print(public_url)

NgrokTunnel: "https://drilling-promptly-stopped.ngrok-free.dev" -> "http://localhost:8000"
